downloading ucimlrepo<br>
since we will be using a health disease dataset from https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators

In [ ]:
!pip install ucimlrepo

importing the modules.

In [ ]:
import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

## importing the dataset from the site.

In [ ]:
# fetch dataset
cdc_diabetes = fetch_ucirepo(id=891)

# data (as pandas dataframes)
X = cdc_diabetes.data.features
y = cdc_diabetes.data.targets
df = pd.concat([X, y], axis=1)

# metadata
print(cdc_diabetes.metadata)
# variable information
print(cdc_diabetes.variables)

{'uci_id': 891, 'name': 'CDC Diabetes Health Indicators', 'repository_url': 'https://archive.ics.uci.edu/dataset/891/cdc+diabetes+health+indicators', 'data_url': 'https://archive.ics.uci.edu/static/public/891/data.csv', 'abstract': 'The Diabetes Health Indicators Dataset contains healthcare statistics and lifestyle survey information about people in general along with their diagnosis of diabetes. The 35 features consist of some demographics, lab test results, and answers to survey questions for each patient. The target variable for classification is whether a patient has diabetes, is pre-diabetic, or healthy. ', 'area': 'Health and Medicine', 'tasks': ['Classification'], 'characteristics': ['Tabular', 'Multivariate'], 'num_instances': 253680, 'num_features': 21, 'feature_types': ['Categorical', 'Integer'], 'demographics': ['Sex', 'Age', 'Education Level', 'Income'], 'target_col': ['Diabetes_binary'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_

In [ ]:
print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype
---  ------                --------------   -----
 0   HighBP                253680 non-null  int64
 1   HighChol              253680 non-null  int64
 2   CholCheck             253680 non-null  int64
 3   BMI                   253680 non-null  int64
 4   Smoker                253680 non-null  int64
 5   Stroke                253680 non-null  int64
 6   HeartDiseaseorAttack  253680 non-null  int64
 7   PhysActivity          253680 non-null  int64
 8   Fruits                253680 non-null  int64
 9   Veggies               253680 non-null  int64
 10  HvyAlcoholConsump     253680 non-null  int64
 11  AnyHealthcare         253680 non-null  int64
 12  NoDocbcCost           253680 non-null  int64
 13  GenHlth               253680 non-null  int64
 14  MentHlth              253680 non-null  int64
 15  PhysHlth              253680 non-n

## we will now clean and process the data.

In [ ]:
#checking if any column has mission values
print(df.isnull().sum())

HighBP                  0
HighChol                0
CholCheck               0
BMI                     0
Smoker                  0
Stroke                  0
HeartDiseaseorAttack    0
PhysActivity            0
Fruits                  0
Veggies                 0
HvyAlcoholConsump       0
AnyHealthcare           0
NoDocbcCost             0
GenHlth                 0
MentHlth                0
PhysHlth                0
DiffWalk                0
Sex                     0
Age                     0
Education               0
Income                  0
Diabetes_binary         0
dtype: int64


#### no columns have missing values, so no imputation is needed here.

## Aggregation.

In [ ]:
aggregation = df.groupby('Diabetes_binary').agg({
    'BMI': 'mean',
    'Age': 'mean',
    'HighBP': 'mean',
    'HighChol': 'mean'
})

print(aggregation)

                       BMI       Age    HighBP  HighChol
Diabetes_binary                                         
0                27.805770  7.814065  0.376602  0.384297
1                31.944011  9.379053  0.752674  0.670118


In [ ]:
aggregation = df.groupby('Diabetes_binary').agg({
    'BMI': ['mean', 'min', 'max'],
    'Age': ['mean', 'median'],
    'PhysHlth': ['mean', 'max']
})

print(aggregation)

                       BMI               Age         PhysHlth    
                      mean min max      mean median      mean max
Diabetes_binary                                                  
0                27.805770  12  98  7.814065    8.0  3.641082  30
1                31.944011  13  98  9.379053   10.0  7.954479  30


#Discretization

In [ ]:
bmi_bins = [0, 18.5, 25, 30, 100]

bmi_labels = [
    'Underweight',
    'Normal',
    'Overweight',
    'Obese'
]

df['bmi_category'] = pd.cut(
    df['BMI'],
    bins=bmi_bins,
    labels=bmi_labels
)

print(df[['BMI', 'bmi_category']].head())

   BMI bmi_category
0   40        Obese
1   25       Normal
2   28   Overweight
3   27   Overweight
4   24       Normal


#Binarization

In [ ]:
df['high_bmi'] = (df['BMI'] >= 30).astype(int)
df['poor_genhlth'] = (df['GenHlth'] >= 4).astype(int)
df['many_bad_mental_days'] = (df['MentHlth'] >= 15).astype(int)
print(df[['BMI', 'high_bmi', 'GenHlth', 'poor_genhlth', 'MentHlth', 'many_bad_mental_days']].head())

   BMI  high_bmi  GenHlth  poor_genhlth  MentHlth  many_bad_mental_days
0   40         1        5             1        18                     1
1   25         0        3             0         0                     0
2   28         0        5             1        30                     1
3   27         0        2             0         0                     0
4   24         0        2             0         3                     0


#Sampling
selecting a subset of the entire dataset.

In [ ]:
sample = df.sample(
    n=100,
    random_state=42
)

print(sample)

        HighBP  HighChol  CholCheck  BMI  Smoker  Stroke  \
219620       0         0          1   21       0       0   
132821       1         1          1   28       0       0   
151862       0         0          1   24       0       0   
139717       0         0          1   27       1       0   
239235       0         1          1   31       1       0   
...        ...       ...        ...  ...     ...     ...   
22059        1         1          1   31       1       0   
1696         1         0          1   31       0       0   
189874       0         0          1   24       1       0   
131080       1         0          1   27       1       0   
100564       1         1          1   24       0       0   

        HeartDiseaseorAttack  PhysActivity  Fruits  Veggies  ...  DiffWalk  \
219620                     0             0       1        1  ...         0   
132821                     0             1       1        1  ...         0   
151862                     0             1   